In [1]:
from mc_experiment import (
    make_seed_counter,
    next_seed,
    standardize_innovations,
    summarize_reference_experiment,
    summarize_mle_augmentation_experiment,
    augmented_config_path,
    size_adjusted_power_comparison,
)

from SymbolicDSGE import ModelParser, DSGESolver, Shock
from SymbolicDSGE.bayesian import make_prior

from numpy import log
import numpy as np

from scipy.stats import chi2, gaussian_kde, norm

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import cProfile

import contextlib
import io
STDOUT_VOID = lambda: contextlib.redirect_stdout(io.StringIO())

_KNOWN_R = False
_AUGMENTED_PARAM = 'r_coef'
_AUGMENTED_EQUATION = 'Rate'
_AUGMENTED_CONFIG = augmented_config_path(_AUGMENTED_EQUATION)
_MEAS_ERR_SCALE = 0.25
_MC_SAMPLES = 1000
_MC_ALPHA = 0.05
_FIGSIZE_1D = (10, 6)
_FIGSIZE_2D = (12, 6)


Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


In [2]:
# Load reference model
parser = ModelParser("../../MODELS/misspec_test/reference.yaml")
config, kalman = parser.get_all()
solver = DSGESolver(config, kalman)

comp = solver.compile(
    n_state=3,
    n_exog=3,
)
sol = solver.solve(
    comp,
    steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
)

print("Transition matrix:\n", sol.A.round(3), "\n")
print("Shock Loadings:\n", sol.B.round(3))

Transition matrix:
 [[ 0.83   0.     0.     0.     0.   ]
 [ 0.     0.85   0.     0.     0.   ]
 [ 0.288 -0.047  0.28   0.     0.   ]
 [ 0.892  0.708 -1.711  0.     0.   ]
 [ 0.7   -0.115 -1.363  0.     0.   ]] 

Shock Loadings:
 [[ 1.     0.     0.   ]
 [ 0.     1.     0.   ]
 [ 0.     0.     1.   ]
 [ 3.193  0.493 -6.107]
 [ 2.531 -0.406 -4.864]]


In [3]:
# Load Misspecified DGP
parser_dgp = ModelParser("../../MODELS/misspec_test/misspec.yaml")
config_dgp, kalman_dgp = parser_dgp.get_all()
solver_dgp = DSGESolver(config_dgp, kalman_dgp)
comp_dgp = solver_dgp.compile(
    n_state=3,
    n_exog=3,
)
sol_dgp = solver_dgp.solve(
    comp_dgp,
    steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
)

In [4]:
# Large sample simulations used to approximate measurement-noise variances
_large_sample_seed_counter = make_seed_counter(start=100_000)
shocks_large = {
    "g,z": Shock(10_000, "norm", multivar=True, seed=next_seed(_large_sample_seed_counter)).shock_generator(),
    "r": Shock(10_000, "norm", multivar=False, seed=next_seed(_large_sample_seed_counter)).shock_generator(),
}

sim1 = sol_dgp.sim(
    T=10_000,
    shocks=shocks_large,
    observables=True,
)

sim2 = sol.sim(
    T=10_000,
    shocks=shocks_large,
    observables=True,
)

In [5]:
T = 200
_plot_seed_counter = make_seed_counter(start=2_000_000)

err_var = np.var(np.column_stack([sim1["OutGap"], sim1["Infl"], sim1["Rate"]]), axis=0)
mc_reference = summarize_reference_experiment(
    sol,
    sol_dgp,
    T=T,
    err_var=err_var,
    meas_err_scale=_MEAS_ERR_SCALE,
    mc_samples=_MC_SAMPLES,
    known_r=_KNOWN_R,
    alpha=_MC_ALPHA,
)

rep_ref = mc_reference["representative"]
sim_dgp = rep_ref.sim_dgp
obs = rep_ref.obs
kf = rep_ref.kf
std_innov = rep_ref.std_innov
err_scale = rep_ref.err_scale
N, n_obs = kf.innov.shape

_measurement_order = {"OutGap": 0, "Infl": 1, "Rate": 2}
_predictor_order = {"Pi": 0, "x": 1, "r": 2}

def _sort_summary(df):
    out = df.copy()
    if "measurement" in out.columns:
        out["measurement_order"] = out["measurement"].map(_measurement_order)
    if "predictor" in out.columns:
        out["predictor_order"] = out["predictor"].map(_predictor_order)
    if "target" in out.columns:
        out["target_order"] = out["target"].map(_predictor_order)
    if "regressor" in out.columns:
        out["regressor_order"] = out["regressor"].map(_predictor_order)
    sort_cols = [
        col
        for col in ["measurement_order", "target_order", "predictor_order", "regressor_order"]
        if col in out.columns
    ]
    if sort_cols:
        out = out.sort_values(sort_cols)
    return out.drop(columns=[col for col in ["measurement_order", "target_order", "predictor_order", "regressor_order"] if col in out.columns])

sim_ref = sol.sim(
    T=T,
    shocks={
        "g,z": Shock(T, "norm", multivar=True, seed=next_seed(_plot_seed_counter)).shock_generator(),
        "r": Shock(T, "norm", multivar=False, seed=next_seed(_plot_seed_counter)).shock_generator(),
    },
    observables=True,
)
ref = np.column_stack([sim_ref["OutGap"], sim_ref["Infl"], sim_ref["Rate"]])[1:, :]

obs_dgp = np.column_stack([sim1["OutGap"], sim1["Infl"], sim1["Rate"]])[1:, :]
if np.any(err_scale != 0.0):
    _plot_rng = np.random.default_rng(next_seed(_plot_seed_counter))
    obs_dgp = obs_dgp + _plot_rng.normal(scale=np.sqrt(err_scale), size=obs_dgp.shape)

In [6]:
print(f"Known R assumption: {_KNOWN_R}")
print(f"Augmented measurement equation: {_AUGMENTED_EQUATION}")
print(f"Augmented coefficient: {_AUGMENTED_PARAM}")
print(f"Monte Carlo replications: {_MC_SAMPLES}")
print("Noise Covariance:\n", np.diag(err_scale).round(3))

Known R assumption: False
Augmented measurement equation: Rate
Augmented coefficient: r_coef
Monte Carlo replications: 1000
Noise Covariance:
 [[3.021 0.    0.   ]
 [0.    4.196 0.   ]
 [0.    0.    0.195]]


In [7]:
print(f"Monte Carlo Ljung-Box summary across {_MC_SAMPLES} replications:")
display(mc_reference["lb_summary"].round(3))

Monte Carlo Ljung-Box summary across 1000 replications:


,measurement,lb_stat,p_value,mc_se_lb_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,23.560,0.0,0.248,0.0,1000,1000,1.0,0.0,0.996,1.0
1,Infl,36.657,0.0,0.288,0.0,1000,1000,1.0,0.0,0.996,1.0
2,Rate,34.266,0.0,0.291,0.0,1000,1000,1.0,0.0,0.996,1.0


In [8]:
print(f"Moment Tests summary across {_MC_SAMPLES} replications:")
display(mc_reference["moment_specification_test_summary"].round(3))

Moment Tests summary across 1000 replications:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.131,1.893,0.656,0.002,0.057,0.008,1000,12,0.012,0.003,0.007,0.021,3.0,200,4
1,cov_identity,28.226,164.961,0.000,0.102,1.124,0.000,1000,1000,1.000,0.000,0.996,1.000,6.0,200,4


In [9]:
print("Innovations on orthogonalized predicted states (Monte Carlo averages and rejection rates):")
_sort_summary(mc_reference["measurement_regressions_orthogonalized_summary"]).round(3)

Innovations on orthogonalized predicted states (Monte Carlo averages and rejection rates):


,measurement,predictor,coef,standardized_coef,std_error,t_stat,p_value,r2,mc_se_coef,mc_se_standardized_coef,mc_se_std_error,mc_se_t_stat,mc_se_p_value,mc_se_r2,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,Pi,0.884,0.053,1.189,0.746,0.425,0.007,0.035,0.002,0.003,0.030,0.009,0.000,1000,101,0.101,0.010,0.084,0.121
1,OutGap,x,-0.722,-0.364,0.131,-5.531,0.000,0.136,0.004,0.002,0.000,0.033,0.000,0.001,1000,1000,1.000,0.000,0.996,1.000
2,OutGap,r,-0.799,-0.030,1.850,-0.422,0.476,0.006,0.057,0.002,0.005,0.031,0.009,0.000,1000,60,0.060,0.008,0.047,0.076
3,Infl,Pi,-1.553,-0.085,1.283,-1.204,0.317,0.012,0.041,0.002,0.002,0.032,0.009,0.000,1000,223,0.223,0.013,0.198,0.250
4,Infl,x,0.119,0.055,0.152,0.782,0.409,0.008,0.005,0.002,0.000,0.033,0.010,0.000,1000,125,0.125,0.010,0.106,0.147
5,Infl,r,-0.801,-0.028,2.003,-0.393,0.466,0.006,0.065,0.002,0.006,0.033,0.009,0.000,1000,75,0.075,0.008,0.060,0.093
6,Rate,Pi,-0.339,-0.087,0.275,-1.242,0.312,0.012,0.008,0.002,0.001,0.030,0.009,0.000,1000,221,0.221,0.013,0.196,0.248
7,Rate,x,0.020,0.043,0.033,0.614,0.435,0.007,0.001,0.002,0.000,0.031,0.009,0.000,1000,78,0.078,0.008,0.063,0.096
8,Rate,r,-1.687,-0.277,0.411,-4.082,0.004,0.081,0.015,0.002,0.001,0.031,0.001,0.001,1000,983,0.983,0.004,0.973,0.989


In [10]:
print("Innovations on raw predicted states (Monte Carlo averages and rejection rates):")
_sort_summary(mc_reference["measurement_regressions_raw_summary"]).round(3)

Innovations on raw predicted states (Monte Carlo averages and rejection rates):


,measurement,predictor,coef,standardized_coef,std_error,t_stat,p_value,r2,mc_se_coef,mc_se_standardized_coef,mc_se_std_error,mc_se_t_stat,mc_se_p_value,mc_se_r2,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
2,OutGap,Pi,-2.762,-0.208,0.916,-3.006,0.026,0.047,0.028,0.002,0.002,0.029,0.002,0.001,1000,868,0.868,0.011,0.846,0.888
1,OutGap,x,-0.646,-0.422,0.098,-6.583,0.000,0.180,0.003,0.002,0.000,0.030,0.000,0.001,1000,1000,1.000,0.000,0.996,1.000
0,OutGap,r,1.739,0.070,1.752,0.993,0.394,0.008,0.047,0.002,0.005,0.026,0.010,0.000,1000,122,0.122,0.010,0.103,0.144
5,Infl,Pi,-0.912,-0.062,1.010,-0.887,0.388,0.009,0.033,0.002,0.001,0.032,0.009,0.000,1000,150,0.150,0.011,0.129,0.173
4,Infl,x,0.028,0.018,0.117,0.251,0.495,0.005,0.004,0.002,0.000,0.032,0.009,0.000,1000,63,0.063,0.008,0.050,0.080
3,Infl,r,-1.322,-0.049,1.898,-0.688,0.423,0.007,0.061,0.002,0.005,0.032,0.009,0.000,1000,108,0.108,0.010,0.090,0.129
8,Rate,Pi,-0.288,-0.095,0.216,-1.354,0.275,0.013,0.006,0.002,0.000,0.030,0.008,0.000,1000,246,0.246,0.014,0.220,0.274
7,Rate,x,0.018,0.049,0.025,0.702,0.437,0.007,0.001,0.002,0.000,0.030,0.010,0.000,1000,94,0.094,0.009,0.077,0.114
6,Rate,r,-1.770,-0.307,0.386,-4.568,0.002,0.098,0.014,0.002,0.001,0.034,0.000,0.001,1000,989,0.989,0.003,0.980,0.994


In [11]:
print("Innovation decomposition on orthogonalized predicted states (Monte Carlo averages):")
_sort_summary(mc_reference["innovation_decomposition_orthogonalized_summary"]).round(3)

Innovation decomposition on orthogonalized predicted states (Monte Carlo averages):


,measurement,predictor,beta_measurement_error,beta_state_prediction_error,beta_total_innovation,beta_component_sum,beta_component_gap,abs_beta_component_gap,reconstruction_max_abs_error,mc_se_beta_measurement_error,mc_se_beta_state_prediction_error,mc_se_beta_total_innovation,mc_se_beta_component_sum,mc_se_beta_component_gap,mc_se_abs_beta_component_gap,mc_se_reconstruction_max_abs_error
0,OutGap,Pi,1.267,-0.383,0.884,0.884,-0.0,0.0,0.0,0.026,0.016,0.035,0.035,0.0,0.0,0.0
1,OutGap,x,0.053,-0.775,-0.722,-0.722,-0.0,0.0,0.0,0.003,0.002,0.004,0.004,0.0,0.0,0.0
2,OutGap,r,-0.312,-0.487,-0.799,-0.799,0.0,0.0,0.0,0.039,0.031,0.057,0.057,0.0,0.0,0.0
3,Infl,Pi,-0.052,-1.500,-1.553,-1.553,-0.0,0.0,0.0,0.019,0.037,0.041,0.041,0.0,0.0,0.0
4,Infl,x,0.001,0.118,0.119,0.119,0.0,0.0,0.0,0.002,0.004,0.005,0.005,0.0,0.0,0.0
5,Infl,r,-0.053,-0.748,-0.801,-0.801,0.0,0.0,0.0,0.030,0.060,0.065,0.065,0.0,0.0,0.0
6,Rate,Pi,0.012,-0.350,-0.339,-0.339,-0.0,0.0,0.0,0.004,0.007,0.008,0.008,0.0,0.0,0.0
7,Rate,x,-0.000,0.020,0.020,0.020,0.0,0.0,0.0,0.000,0.001,0.001,0.001,0.0,0.0,0.0
8,Rate,r,-0.029,-1.659,-1.687,-1.687,0.0,0.0,0.0,0.006,0.013,0.015,0.015,0.0,0.0,0.0


In [12]:
print("Innovation decomposition on raw predicted states (Monte Carlo averages):")
_sort_summary(mc_reference["innovation_decomposition_raw_summary"]).round(3)

Innovation decomposition on raw predicted states (Monte Carlo averages):


,measurement,predictor,beta_measurement_error,beta_state_prediction_error,beta_total_innovation,beta_component_sum,beta_component_gap,abs_beta_component_gap,reconstruction_max_abs_error,mc_se_beta_measurement_error,mc_se_beta_state_prediction_error,mc_se_beta_total_innovation,mc_se_beta_component_sum,mc_se_beta_component_gap,mc_se_abs_beta_component_gap,mc_se_reconstruction_max_abs_error
0,OutGap,Pi,1.552,-4.314,-2.762,-2.762,0.0,0.0,0.0,0.020,0.016,0.028,0.028,0.0,0.0,0.0
1,OutGap,x,0.146,-0.792,-0.646,-0.646,0.0,0.0,0.0,0.002,0.002,0.003,0.003,0.0,0.0,0.0
2,OutGap,r,-0.367,2.107,1.739,1.739,0.0,0.0,0.0,0.041,0.035,0.047,0.047,0.0,0.0,0.0
3,Infl,Pi,-0.041,-0.871,-0.912,-0.912,0.0,0.0,0.0,0.014,0.030,0.033,0.033,0.0,0.0,0.0
4,Infl,x,-0.003,0.030,0.028,0.028,-0.0,0.0,0.0,0.002,0.003,0.004,0.004,0.0,0.0,0.0
5,Infl,r,-0.056,-1.266,-1.322,-1.322,0.0,0.0,0.0,0.028,0.057,0.061,0.061,0.0,0.0,0.0
6,Rate,Pi,0.009,-0.297,-0.288,-0.288,0.0,0.0,0.0,0.003,0.006,0.006,0.006,0.0,0.0,0.0
7,Rate,x,0.001,0.017,0.018,0.018,0.0,0.0,0.0,0.000,0.001,0.001,0.001,0.0,0.0,0.0
8,Rate,r,-0.023,-1.748,-1.770,-1.770,0.0,0.0,0.0,0.006,0.013,0.014,0.014,0.0,0.0,0.0


Monte Carlo summaries above aggregate `_MC_SAMPLES` independent draws. The plots and MCMC output below continue on a representative first draw so the original visual workflow remains available.


In [13]:
parser_aug = ModelParser(_AUGMENTED_CONFIG)
config_aug, kalman_aug = parser_aug.get_all()
solver_aug = DSGESolver(config_aug, kalman_aug)
comp_aug = solver_aug.compile(
    n_state=3,
    n_exog=3,
)
priors = {
    _AUGMENTED_PARAM: make_prior(
        'normal',
        parameters={"mean": 0.0, "std": 4.0, "random_state": next_seed(_plot_seed_counter)},
        transform="identity",
    ),
}

with STDOUT_VOID():
    mc_aug = summarize_mle_augmentation_experiment(
        sol,
        solver_aug,
        comp_aug,
        sol_dgp,
        mc_reference,
        T=T,
        candidate_param=_AUGMENTED_PARAM,
        mc_samples=_MC_SAMPLES,
        alpha=_MC_ALPHA,
    )

# estim = lambda: solver_aug.estimate_and_solve(
#     compiled=comp_aug,
#     method="mcmc",
#     n_draws=25_000,
#     burn_in=10_000,
#     thin=2,
#     posterior_point="mean",
#     proposal_scale=1.0,
#     y=obs,
#     priors=priors,
#     steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
#     random_state=next_seed(_plot_seed_counter),
#     **mc_reference["filter_kwargs"],
# )
# res_aug, sol_aug = estim()

## Diagnostics of the Augmented Model

The figures below still display the representative first draw. The scalar summaries reported in later cells are Monte Carlo averages.


### Marginal LR Test Conditional on $	heta_0$

The table below reports the Monte Carlo MLE summary for the LR test.


In [14]:
print("Monte Carlo LR summary for the MLE-augmented model:")
rep_aug = mc_aug["representative"]
res_mle = rep_aug.res_mle
sol_mle = rep_aug.sol_mle
mle_aug_kf = rep_aug.kf_aug
std_innov_aug_mle = rep_aug.std_innov_aug
sim_aug_mle = rep_aug.sim_aug
mc_aug["lr_summary"].round(3)

Monte Carlo LR summary for the MLE-augmented model:


,estimated_coef,loglik_ref,loglik_aug,lr,p_value,mc_se_estimated_coef,mc_se_loglik_ref,mc_se_loglik_aug,mc_se_lr,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,0.649,-4103.772,-4077.607,52.33,0.018,0.01,10.591,10.683,1.315,0.003,1000,958,0.958,0.006,0.944,0.969


In [15]:
res_mle

OptimizationResult(kind='mle', x=array([0.83201601]), theta={'beta': np.float64(0.971), 'kappa': np.float64(0.58), 'tau_inv': np.float64(1.86), 'psi_pi': np.float64(2.19), 'psi_x': np.float64(0.3), 'rho_r': np.float64(0.84), 'rho_g': np.float64(0.83), 'rho_z': np.float64(0.85), 'pi_star': np.float64(3.43), 'r_star': np.float64(3.01), 'sig_r': np.float64(0.18), 'sig_g': np.float64(0.18), 'sig_z': np.float64(0.64), 'rho_gz': np.float64(0.36), 'meas_infl': np.float64(1e-06), 'meas_rate': np.float64(1e-06), 'meas_outgap': np.float64(1e-06), 'meas_rho_ir': np.float64(0.0), 'meas_rho_gi': np.float64(0.0), 'meas_rho_gr': np.float64(0.0), 'Pi_coef': np.float64(0.0), 'x_coef': np.float64(0.0), 'r_coef': np.float64(0.8320160129054287)}, success=True, message='CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH', fun=np.float64(3565.2536021133424), loglik=np.float64(-3565.2536021133424), logprior=np.float64(0.0), logpost=np.float64(-3565.2536021133424), nfev=12, nit=5, raw=  message: CONVERGENCE

## Serial Autocorrelation Tests for the Augmented Model

The figure below uses the representative MCMC draw, while the printed table reports Monte Carlo MLE rejection frequencies.


In [16]:
print("Monte Carlo Ljung-Box summary for the MLE-augmented model:")
display(mc_aug["lb_summary"].round(3))

Monte Carlo Ljung-Box summary for the MLE-augmented model:


,measurement,lb_stat,p_value,mc_se_lb_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,23.554,0.0,0.248,0.0,1000,1000,1.0,0.0,0.996,1.0
1,Infl,36.654,0.0,0.288,0.0,1000,1000,1.0,0.0,0.996,1.0
2,Rate,34.005,0.0,0.289,0.0,1000,1000,1.0,0.0,0.996,1.0


In [17]:
print("Reference moment-specification test summary:")
display(mc_reference["moment_specification_test_summary"].round(3))

print("Augmented moment-specification test summary:")
display(mc_aug["moment_specification_test_summary"].round(3))

print("Reference-minus-augmented moment distance comparison:")
display(mc_aug["moment_specification_comparison"].round(3))

Reference moment-specification test summary:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.131,1.893,0.656,0.002,0.057,0.008,1000,12,0.012,0.003,0.007,0.021,3.0,200,4
1,cov_identity,28.226,164.961,0.000,0.102,1.124,0.000,1000,1000,1.000,0.000,0.996,1.000,6.0,200,4


Augmented moment-specification test summary:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.126,1.866,0.662,0.002,0.057,0.008,1000,13,0.013,0.004,0.008,0.022,3.0,200,4
1,cov_identity,28.458,168.988,0.000,0.103,1.182,0.000,1000,1000,1.000,0.000,0.996,1.000,6.0,200,4


Reference-minus-augmented moment distance comparison:


,test,n_replications,distance_ref,mc_se_distance_ref,distance_aug,mc_se_distance_aug,distance_improvement,mc_se_distance_improvement,stat_ref,mc_se_stat_ref,stat_aug,mc_se_stat_aug,stat_improvement,mc_se_stat_improvement,aug_closer_rate,aug_closer_rate_mc_se,aug_closer_ci_low,aug_closer_ci_high
0,mean_zero_hac,1000,0.131,0.002,0.126,0.002,0.005,0.000,1.893,0.057,1.866,0.057,0.027,0.004,0.838,0.012,0.814,0.860
1,cov_identity,1000,28.226,0.102,28.458,0.103,-0.233,0.003,164.961,1.124,168.988,1.182,-4.027,0.412,0.006,0.002,0.003,0.013
